# 00 – Primeros pasos con pm4py

Este notebook **no resuelve el examen**. Solo muestra cómo:

1. cargar el event log de un caso;
2. hacer una revisión rápida de los datos;
3. convertirlo al formato de pm4py;
4. obtener estadísticas básicas y un primer mapa del proceso (DFG).

Para trabajar, guarde una copia con otro nombre (por ejemplo, `scripts/01_exploracion.ipynb`) usando **Archivo → Guardar una copia en GitHub**, y adáptela a su caso.

> **Importante:** las decisiones de limpieza de este notebook son solo para que el código funcione. **No son la limpieza correcta.** Su pareja debe revisar los datos, decidir qué hacer y justificarlo en el Corte 2.

## 0. Configuración en Google Colab

Ejecute esta celda primero. Cambie `REPO` por el usuario y el nombre del repositorio de la pareja. El token se lee del secreto `GITHUB_TOKEN` (ver la sección *Entorno de trabajo* del README).

In [ ]:
# Configuración del entorno en Google Colab
from google.colab import userdata

REPO = "usuario/nombre-del-repositorio"
TOKEN = userdata.get("GITHUB_TOKEN")

!git clone -q https://{TOKEN}@github.com/{REPO}.git /content/repo
%cd /content/repo/scripts
!pip install -q pm4py

In [ ]:
from pathlib import Path

import pandas as pd
import pm4py

# Cambie el nombre del archivo según el caso seleccionado
CASO = "caso_04_soporte_ti.csv"

# La celda de configuración deja el notebook ubicado en la carpeta scripts/
RAW = Path("..") / "datos" / "raw"
PROCESSED = Path("..") / "datos" / "processed"


## 1. Carga y vista general

In [ ]:
df = pd.read_csv(RAW / CASO, encoding="utf-8")
print(df.shape)
df.head()

In [ ]:
df.info()

## 2. Revisión rápida

Antes de construir el Event Log, responda:

- ¿Las actividades están escritas siempre igual?
- ¿Todas las fechas tienen el mismo formato? ¿Hay fechas vacías?
- ¿Hay filas repetidas?
- ¿Todos los casos terminan en una actividad final?
- ¿Qué columnas describen al **caso** y cuáles al **evento**?

In [ ]:
df["activity"].value_counts()

In [ ]:
# Intento de conversión: los valores que no se puedan convertir quedan como NaT
ts = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
print("Timestamps no convertidos:", ts.isna().sum())
df.loc[ts.isna(), ["case_id", "activity", "timestamp"]].head(10)

In [ ]:
print("Filas duplicadas exactas:", df.duplicated().sum())

## 3. Formato pm4py

pm4py necesita saber qué columna es el caso, cuál la actividad y cuál el timestamp, y que este último sea de tipo fecha.

> En la celda siguiente se **descartan** sin más los eventos con fecha no convertida. Es solo para demostrar el flujo. Revise qué son esos eventos y decida cómo tratarlos.

In [ ]:
demo = df.assign(timestamp=ts).dropna(subset=["timestamp"])

log = pm4py.format_dataframe(demo, case_id="case_id", activity_key="activity", timestamp_key="timestamp")
log = log.sort_values(["case:concept:name", "time:timestamp"])
log[["case:concept:name", "concept:name", "time:timestamp", "resource"]].head()

## 4. Estadísticas básicas

In [ ]:
print("Casos:", log["case:concept:name"].nunique())
print("Eventos:", len(log))
print("Actividades:", log["concept:name"].nunique())
print("Desde:", log["time:timestamp"].min(), " Hasta:", log["time:timestamp"].max())
print("Actividades iniciales:", pm4py.get_start_activities(log))
print("Actividades finales:", pm4py.get_end_activities(log))

In [ ]:
variantes = pm4py.get_variants(log)
print("Número de variantes:", len(variantes))

# Cinco variantes más frecuentes
for variante, n in sorted(variantes.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(n, " → ".join(variante))

In [ ]:
# Duración de cada caso (días), desde su primer hasta su último evento.
# Pregúntese: ¿el último evento es el que define el final del proceso en su caso?
duraciones = pd.Series(pm4py.get_all_case_durations(log)) / 86400
duraciones.describe()

## 5. Primer mapa del proceso (DFG)

El dibujo usa **Graphviz**, que ya viene instalado en Colab. Si no estuviera disponible, la celda muestra las transiciones como tabla.

In [ ]:
dfg, inicio, fin = pm4py.discover_dfg(log)

try:
    pm4py.view_dfg(dfg, inicio, fin, format="png")
except Exception as e:
    print("No se pudo dibujar (¿falta Graphviz?):", type(e).__name__)
    display(pd.Series(dfg, name="frecuencia").sort_values(ascending=False).head(15).to_frame())

In [ ]:
# DFG de rendimiento: tiempo medio entre actividades (en segundos)
perf_dfg, inicio, fin = pm4py.discover_performance_dfg(log)

try:
    pm4py.view_performance_dfg(perf_dfg, inicio, fin, format="png")
except Exception as e:
    print("No se pudo dibujar (¿falta Graphviz?):", type(e).__name__)
    tiempos = {k: v["mean"] / 3600 for k, v in perf_dfg.items()}
    display(pd.Series(tiempos, name="horas_promedio").sort_values(ascending=False).head(15).to_frame())

## 6. Siguientes pasos

- Escriba la limpieza en su propio notebook o script, leyendo de `datos/raw/` y guardando el resultado en `datos/processed/`.
- Si el mapa resulta ilegible, considere filtrar variantes o actividades (`pm4py.filter_variants_top_k`, `pm4py.filter_event_attribute_values`), y documente el filtro.
- Explore otros algoritmos de descubrimiento (`pm4py.discover_petri_net_inductive`, `pm4py.discover_heuristics_net`) y la verificación de cumplimiento (`pm4py.conformance_diagnostics_token_based_replay`).
- Revise [recursos/glosario.md](../recursos/glosario.md) y, antes del Corte 3, [recursos/fuga_de_informacion.md](../recursos/fuga_de_informacion.md).